Extractive Summarisation Approach

Selecting the most important sentences from the original text and combining them to form summary

Nothing new is generated therefore no training is required.

Methods for computing scores and extracing sentences:

1. Frequency based scoring methods
2. Tfidf method
3. Using TextRank

In [ ]:
pip install datasets

In [ ]:
from datasets import load_dataset
dataset = load_dataset("cnn_dailymail","3.0.0")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})


In [ ]:
eg_article = dataset['train'][0]['article']
eg_summ = dataset['train'][0]['highlights']

print(f"Article: {eg_article[:200]}..")
print(f"Summary: {eg_summ}")

Article: LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on ..
Summary: Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


In [ ]:
contractions = {
"ain't": "am not",
"aren't": "are not",
"can't": "cannot",
"can't've": "cannot have",
"'cause": "because",
"could've": "could have",
"couldn't": "could not",
"couldn't've": "could not have",
"didn't": "did not",
"doesn't": "does not",
"don't": "do not",
"hadn't": "had not",
"hadn't've": "had not have",
"hasn't": "has not",
"haven't": "have not",
"he'd": "he would",
"he'd've": "he would have",
"he'll": "he will",
"he's": "he is",
"how'd": "how did",
"how'll": "how will",
"how's": "how is",
"i'd": "i would",
"i'll": "i will",
"i'm": "i am",
"i've": "i have",
"isn't": "is not",
"it'd": "it would",
"it'll": "it will",
"it's": "it is",
"let's": "let us",
"ma'am": "madam",
"mayn't": "may not",
"might've": "might have",
"mightn't": "might not",
"must've": "must have",
"mustn't": "must not",
"needn't": "need not",
"oughtn't": "ought not",
"shan't": "shall not",
"sha'n't": "shall not",
"she'd": "she would",
"she'll": "she will",
"she's": "she is",
"should've": "should have",
"shouldn't": "should not",
"that'd": "that would",
"that's": "that is",
"there'd": "there had",
"there's": "there is",
"they'd": "they would",
"they'll": "they will",
"they're": "they are",
"they've": "they have",
"wasn't": "was not",
"we'd": "we would",
"we'll": "we will",
"we're": "we are",
"we've": "we have",
"weren't": "were not",
"what'll": "what will",
"what're": "what are",
"what's": "what is",
"what've": "what have",
"where'd": "where did",
"where's": "where is",
"who'll": "who will",
"who's": "who is",
"won't": "will not",
"wouldn't": "would not",
"you'd": "you would",
"you'll": "you will",
"you're": "you are"
}

TextRank is:

Unsupervised (no training needed)

Simple and strong baseline for extractive summarization

Works by ranking sentences using graph importance (similar to PageRank)

sent= node,
similarity = edge,
build graph,
appy pagerank,
sent with highest score shown

In [ ]:
import re
import nltk
import string
import numpy as np
import networkx  as nx
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [ ]:
def expand_contractions(text):
  all_keys = '|'.join(contractions.keys())
  pattern = rf"\b({all_keys})\b"
  def replace_match(match):
    word = match.group(0)
    return contractions[word.lower()]
  return re.sub(pattern, replace_match, text, flags = re.IGNORECASE)
#ignore case = for handling I'm or i'm

In [ ]:
def remove_punctuation(text):
  return text.translate(str.maketrans('','',string.punctuation))

In [ ]:
stop_words = set(stopwords.words('english'))
def remove_stopwords(text):
  filtered = []
  for word in text.split():
    if word.lower() not in stop_words:
      filtered.append(word)
  return " ".join(filtered)

In [ ]:
lemma = WordNetLemmatizer()
def lemmatizing(text):
  lemma_list = []
  for word in text.split():
    new_word = lemma.lemmatize(word)
    lemma_list.append(new_word)
  return " ".join(lemma_list)

In [ ]:
#full preprocessing function

def preprocess(sent):
  sent = expand_contractions(sent)
  sent = sent.lower()
  sent = remove_punctuation(sent)
  sent = remove_stopwords(sent)
  sent = lemmatizing(sent)
  return sent

In [ ]:
def process_article(article):
    original_sentences = sent_tokenize(article)

    processed_sentences = []

    for sent in original_sentences:
        clean_sent = preprocess(sent)
        processed_sentences.append(clean_sent)

    return original_sentences, processed_sentences

In [ ]:
article = """
Artificial Intelligence is transforming industries.
Companies are investing heavily in AI research.
These technologies improve healthcare and automation.
"""

# original, processed = process_article(article)

# print("Original Sentences:\n", original)
# print("\nProcessed Sentences:\n", processed)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def build_vectors(processed_sentences):
  vector = TfidfVectorizer()
  sent_vectors = vector.fit_transform(processed_sentences)
  return sent_vectors

In [ ]:
original, processed = process_article(article)
sent_vectors = build_vectors(processed)
#shape of tfidf_mat
print(sent_vectors.shape)

(3, 13)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def build_similarity_matrix(sent_vectors):
  sim_mat = cosine_similarity(sent_vectors)
  return sim_mat

In [ ]:
def rank_summ(sim_mat):
  graph = nx.from_numpy_array(sim_mat)
  scores = nx.pagerank(graph)
  return scores

In [ ]:
def gen_summ(original_sent, scores, k=3):
  ranked_sent = sorted(((scores[i],s,i) for i,s in enumerate(original_sent)), reverse=True)
  selected = sorted(ranked_sent[:k],key=lambda x:x[2])
  summ = [s[1] for s in selected]
  return " ".join(summ)

In [ ]:
sample = dataset['test'][0]

article = sample['article']
refer = sample['highlights']

original, processed = process_article(article)

vec = build_vectors(processed)
sim = build_similarity_matrix(vec)
scores = rank_summ(sim)

summ = gen_summ(original, scores, k=3)
print("refrenece summary:\n", refer)
print("\ntextrank summary:\n",summ)

refrenece summary:
 Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .

textrank summary:
 (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Israel and the United States, neither of which is an ICC member, opposed the Palestinians' efforts to join the body.


In [ ]:
print(type(sample))

<class 'dict'>


In [ ]:
print(f"Sentences found: {len(original)}")
print(f"Scores generated: {len(scores)}")

Sentences found: 27
Scores generated: 27


In [ ]:
print(dataset['test'][0]['article'])

(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki, speaking at Wednesday's ceremony, sa

we can improve quailty of summary,


"if len(sentence.split()) < 5:
    skip"

remove short sent

sim_matrix = cosine_similarity(vectors)
sim_matrix[sim_matrix < 0.1] = 0

prevents weak links
more meaningful

top_n = int(len(sentences) * 0.3)

summary is concise

In [1]:
#when i implemented summariser in which scoring is based on tfidf method
#high tfidf score = are selected

# import numpy as np

# sent_score = np.array(tfidf_mat.mean(axis=1)).flatten()

# k=3;
# top_ind = sent_score.argsort()[-k:]
# top_ind = sorted(top_ind)

# summary = " ".join([original[i] for i in top_ind])
# print(summary)
# for i, s in enumerate(sent_score):
#     print(i, s, original[i])


# for article in dataset['test']['article'][:10]:

#     original, processed = process_article(article)

#     tfidf_mat = vector.transform(processed)

#     scores = np.array(tfidf_mat.sum(axis=1)).flatten()

#     top_ind = scores.argsort()[-3:]
#     top_ind = sorted(top_ind)

#     summary = " ".join([original[i] for i in top_ind])

#     print(summary)